# Projeto – Dashboard Multinível para Tomada de Decisão
**Visualização de Dados e Design de Dashboards – Ciência de Dados para Negócios (FATEC Votorantim)**

**Grupo:**  
Ana Elisa Rubinato Cavalheiro  
Arthur Nunes dos Santos  
Bruno Felipe de Souza Araujo  
Caio Cesar Corrá  
Lucas Camelo  
Nicole Fava Pires

Cada célula de código responde **uma pergunta** (no título da célula). As respostas, KPIs, wireframes, insights e recomendações estão no documento **Documentacao_Projeto_Dashboard_Multinivel.docx**.

## 1. Configuração e carga
Envie `Base_Projeto_VisualizacaodeDados.xlsx` para o Volume. Em outro workspace, altere só `CATALOGO`, `SCHEMA` e `VOLUME` na célula de configuração.

In [0]:
%pip install openpyxl -q

In [0]:
%restart_python

In [0]:
# Configuração: para usar outro workspace, altere apenas estas 3 linhas
CATALOGO = "visualizacaodados"   # catalogo visualizacao de dados criado para disciplina de visualizacao de dados
SCHEMA = "default"
VOLUME = "arquivos"

import pandas as pd
import matplotlib.pyplot as plt

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{SCHEMA}.{VOLUME}")
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Mesma identidade visual em todos os gráficos: cinza = referência, azul = destaque, vermelho = alerta
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
AZUL, VERMELHO, CINZA = "#1f77b4", "#d62728", "#9e9e9e"

print(f"Usando {CATALOGO}.{SCHEMA}")

In [0]:
# Carga: a tabela base_projeto já existe? Se não, cria a partir do arquivo Excel do Volume
ARQUIVO = f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}/Base_Projeto_VisualizacaodeDados.xlsx"

if spark.catalog.tableExists("base_projeto"):
    print("Tabela base_projeto já existe.")
else:
    pdf = pd.read_excel(ARQUIVO, sheet_name="Dados")
    pdf["Motivo_Devolucao"] = pdf["Motivo_Devolucao"].astype(object).where(pdf["Motivo_Devolucao"].notna(), None)
    spark.createDataFrame(pdf).write.mode("overwrite").saveAsTable("base_projeto")
    print("Tabela base_projeto criada a partir do arquivo.")

display(spark.table("base_projeto").limit(10))

## 2. Auditoria dos dados

In [0]:
%sql
-- 2.1 Há pedidos duplicados? Qual o período da base?
SELECT
    COUNT(*) AS Total_Registros,
    COUNT(DISTINCT ID_Pedido) AS Pedidos_Unicos,
    COUNT(*) - COUNT(DISTINCT ID_Pedido) AS IDs_Duplicados,
    MIN(Data_Pedido) AS Data_Inicial,
    MAX(Data_Pedido) AS Data_Final
FROM base_projeto

In [0]:
# 2.2 Quais colunas têm valores ausentes? Há linhas inteiras repetidas?
base = spark.table("base_projeto").toPandas()

ausentes = base.isna().sum()

print("Valores ausentes por coluna:")
print(ausentes)

print("\nLinhas totalmente duplicadas:", base.duplicated().sum())

In [0]:
%sql
-- 2.3 Ano, Mês e Trimestre batem com a data do pedido?
SELECT
    SUM(CASE WHEN Ano <> YEAR(Data_Pedido) THEN 1 ELSE 0 END) AS Ano_Divergente,
    SUM(CASE WHEN Mes <> MONTH(Data_Pedido) THEN 1 ELSE 0 END) AS Mes_Divergente,
    SUM(CASE WHEN Trimestre <> CONCAT('T', QUARTER(Data_Pedido)) THEN 1 ELSE 0 END) AS Trimestre_Divergente,
    SUM(CASE WHEN Data_Pedido IS NULL THEN 1 ELSE 0 END) AS Data_Nula
FROM base_projeto

In [0]:
# 2.4 Há categorias escritas de formas diferentes (espaços, maiúsculas/minúsculas)?
textos = ["Regiao", "Estado", "Cidade", "Canal_Venda", "Segmento_Cliente", "Categoria",
          "Subcategoria", "Produto", "Vendedor", "Equipe", "Status_Pedido"]

categorias = pd.DataFrame({
    "Valores_Distintos": base[textos].nunique(),
    "Distintos_Apos_Padronizar": base[textos].apply(lambda c: c.str.strip().str.upper().nunique()),
})
display(categorias.reset_index(names="Campo"))

# Hierarquia geográfica: algum estado aparece em mais de uma região?
regioes_por_estado = base.groupby("Estado")["Regiao"].nunique()
print("Estados em mais de uma região:", (regioes_por_estado > 1).sum())

In [0]:
# 2.5 A Receita Bruta é igual a Quantidade x Preço? Se não, em quais meses?
base["Fator"] = base["Receita_Bruta"] / (base["Quantidade"] * base["Preco_Unitario"])
diferentes = (base["Receita_Bruta"] - base["Quantidade"] * base["Preco_Unitario"]).abs() > 1
print("Pedidos em que Receita Bruta ≠ Quantidade x Preço:", diferentes.sum())

fator_mes = base.groupby("Mes")["Fator"].mean().round(2)
cores = [CINZA if f == 1 else VERMELHO for f in fator_mes]
fator_mes.plot.bar(color=cores, rot=0, title="Fator médio Receita Bruta / (Quantidade x Preço), por mês")
plt.axhline(1, color="black", linewidth=0.8)
plt.show()

In [0]:
%sql
-- 2.6 Receita Líquida = Bruta x (1 - Desconto)? Lucro = Líquida - Custo? Margem = Lucro / Líquida?
SELECT
    Status_Pedido,
    COUNT(*) AS Pedidos,
    SUM(CASE WHEN ABS(Receita_Liquida - Receita_Bruta * (1 - Desconto_Pct)) > 1 THEN 1 ELSE 0 END) AS Liquida_Divergente,
    SUM(CASE WHEN ABS(Lucro - (Receita_Liquida - Custo_Total)) > 1 THEN 1 ELSE 0 END) AS Lucro_Divergente,
    SUM(CASE WHEN Receita_Liquida > 0 AND ABS(Margem_Pct - Lucro / Receita_Liquida) > 0.001 THEN 1 ELSE 0 END) AS Margem_Divergente,
    SUM(CASE WHEN Receita_Liquida = 0 THEN 1 ELSE 0 END) AS Receita_Liquida_Zero,
    ROUND(SUM(Receita_Bruta), 2) AS Receita_Bruta_Total
FROM base_projeto
GROUP BY Status_Pedido

In [0]:
# 2.7 Quais são as faixas de valores (mínimo, máximo)? Há pedidos com lucro negativo?
colunas = ["Quantidade", "Preco_Unitario", "Desconto_Pct", "Satisfacao_Cliente", "Atraso_Dias", "Lucro"]
display(base[colunas].describe().round(2).reset_index(names="Estatistica"))

prejuizo = base[base["Lucro"] < 0]
print(f"Pedidos com lucro negativo: {len(prejuizo)} | Prejuízo total: R$ {prejuizo['Lucro'].sum():,.2f}")

In [0]:
# 2.8 Existem valores extremos (outliers)? Regra IQR: acima de Q3 + 1,5 x (Q3 - Q1)
validos = base[base["Status_Pedido"] != "Cancelado"]

for coluna in ["Quantidade", "Receita_Liquida"]:
    q1, q3 = validos[coluna].quantile([0.25, 0.75])
    limite = q3 + 1.5 * (q3 - q1)
    print(f"{coluna}: limite = {limite:,.1f} | pedidos acima do limite = {(validos[coluna] > limite).sum()}")

validos.boxplot(column="Receita_Liquida", by="Segmento_Cliente", grid=False)
plt.suptitle("")
plt.title("Distribuição da receita líquida por pedido e segmento (pontos = outliers)")
plt.show()

In [0]:
%sql
-- 2.9 Há pedidos antigos ainda "Em processamento"? Devolvido x Motivo é coerente?
SELECT
    YEAR(Data_Pedido) AS Ano,
    SUM(CASE WHEN Status_Pedido = 'Em processamento' THEN 1 ELSE 0 END) AS Em_Processamento,
    SUM(CASE WHEN Status_Pedido = 'Cancelado' THEN 1 ELSE 0 END) AS Cancelados,
    SUM(CASE WHEN Devolvido = 'Sim' AND Motivo_Devolucao IS NULL THEN 1 ELSE 0 END) AS Devolvido_Sem_Motivo,
    SUM(CASE WHEN Devolvido = 'Não' AND Motivo_Devolucao IS NOT NULL THEN 1 ELSE 0 END) AS Motivo_Sem_Devolucao
FROM base_projeto
GROUP BY YEAR(Data_Pedido)
ORDER BY Ano

## 3. Tratamento e novas variáveis

In [0]:
%sql
-- 3.1 Cria a tabela tratada: limpeza + novas variáveis
CREATE OR REPLACE TABLE base_projeto_tratada AS
WITH base AS (
    SELECT
        TRIM(ID_Pedido) AS ID_Pedido,
        CAST(Data_Pedido AS DATE) AS Data_Pedido,
        TRIM(Regiao) AS Regiao,
        UPPER(TRIM(Estado)) AS Estado,
        TRIM(Cidade) AS Cidade,
        TRIM(Canal_Venda) AS Canal_Venda,
        TRIM(Segmento_Cliente) AS Segmento_Cliente,
        TRIM(ID_Cliente) AS ID_Cliente,
        TRIM(Categoria) AS Categoria,
        TRIM(Subcategoria) AS Subcategoria,
        TRIM(Produto) AS Produto,
        TRIM(Vendedor) AS Vendedor,
        TRIM(Equipe) AS Equipe,
        CAST(Quantidade AS INT) AS Quantidade,
        CAST(Preco_Unitario AS DOUBLE) AS Preco_Unitario,
        CAST(Desconto_Pct AS DOUBLE) AS Desconto_Pct,
        CAST(Receita_Bruta AS DOUBLE) AS Receita_Bruta,
        CAST(Receita_Liquida AS DOUBLE) AS Receita_Liquida,
        CAST(Custo_Total AS DOUBLE) AS Custo_Total,
        CAST(Lucro AS DOUBLE) AS Lucro,
        CAST(Margem_Pct AS DOUBLE) AS Margem_Pct,
        CAST(Meta_Receita AS DOUBLE) AS Meta_Receita,
        CAST(Prazo_Entrega_Dias AS INT) AS Prazo_Entrega_Dias,
        CAST(Atraso_Dias AS INT) AS Atraso_Dias,
        CAST(Satisfacao_Cliente AS DOUBLE) AS Satisfacao_Cliente,
        TRIM(Devolvido) AS Devolvido,
        COALESCE(NULLIF(TRIM(Motivo_Devolucao), ''), 'Sem devolução') AS Motivo_Devolucao,
        TRIM(Status_Pedido) AS Status_Pedido
    FROM base_projeto
),
limites AS (
    -- limites IQR dos outliers e data mais recente da base
    SELECT
        percentile_approx(Quantidade, 0.75) + 1.5 * (percentile_approx(Quantidade, 0.75) - percentile_approx(Quantidade, 0.25)) AS Limite_Quantidade,
        percentile_approx(Receita_Liquida, 0.75) + 1.5 * (percentile_approx(Receita_Liquida, 0.75) - percentile_approx(Receita_Liquida, 0.25)) AS Limite_Receita,
        MAX(Data_Pedido) AS Data_Referencia
    FROM base
    WHERE Status_Pedido <> 'Cancelado'
),
abc AS (
    -- % acumulado da receita por produto (do maior para o menor) para a curva ABC
    SELECT
        Produto,
        SUM(SUM(Receita_Liquida)) OVER (ORDER BY SUM(Receita_Liquida) DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            / SUM(SUM(Receita_Liquida)) OVER () AS Perc_Acumulado
    FROM base
    GROUP BY Produto
)
SELECT
    b.*,
    -- calendário
    YEAR(b.Data_Pedido) AS Ano,
    CAST(YEAR(b.Data_Pedido) AS STRING) AS Ano_Texto,
    MONTH(b.Data_Pedido) AS Mes,
    CONCAT('T', QUARTER(b.Data_Pedido)) AS Trimestre,
    CAST(DATE_TRUNC('MONTH', b.Data_Pedido) AS DATE) AS Ano_Mes,
    CASE WHEN MONTH(b.Data_Pedido) <= MONTH(l.Data_Referencia) THEN 1 ELSE 0 END AS Periodo_Comparavel,
    -- indicadores 1/0
    CASE WHEN b.Status_Pedido = 'Cancelado' THEN 0 ELSE 1 END AS Pedido_Valido,
    CASE WHEN b.Status_Pedido = 'Cancelado' THEN 1 ELSE 0 END AS Flag_Cancelado,
    CASE WHEN b.Devolvido = 'Sim' THEN 1 ELSE 0 END AS Flag_Devolvido,
    CASE WHEN b.Status_Pedido <> 'Cancelado' AND b.Atraso_Dias > 0 THEN 1 ELSE 0 END AS Flag_Atraso,
    CASE WHEN b.Lucro < 0 THEN 1 ELSE 0 END AS Flag_Prejuizo,
    -- valores derivados
    CASE WHEN b.Status_Pedido = 'Cancelado' THEN 0 ELSE b.Meta_Receita END AS Meta_Valida,
    CASE WHEN b.Status_Pedido = 'Cancelado' THEN b.Receita_Bruta ELSE 0 END AS Receita_Cancelada,
    CASE WHEN b.Status_Pedido = 'Cancelado' THEN 0 ELSE b.Receita_Bruta END AS Receita_Bruta_Valida,
    CASE WHEN b.Status_Pedido = 'Cancelado' THEN 0 ELSE ROUND(b.Receita_Bruta - b.Receita_Liquida, 2) END AS Valor_Desconto,
    CASE WHEN b.Lucro < 0 THEN b.Lucro ELSE 0 END AS Valor_Prejuizo,
    ROUND(b.Margem_Pct * 100, 2) AS Margem_Perc,
    ROUND(b.Desconto_Pct * 100, 0) AS Desconto_Perc,
    -- sazonalidade
    ROUND(b.Receita_Bruta / (b.Quantidade * b.Preco_Unitario), 2) AS Fator_Sazonal,
    CASE WHEN MONTH(b.Data_Pedido) IN (1, 2) THEN 'Baixa (jan-fev)'
         WHEN MONTH(b.Data_Pedido) IN (11, 12) THEN 'Alta (nov-dez)'
         ELSE 'Normal (mar-out)' END AS Periodo_Sazonal,
    -- faixas e classificação
    CASE WHEN b.Desconto_Pct <= 0.05 THEN '1. Até 5%'
         WHEN b.Desconto_Pct <= 0.10 THEN '2. 6% a 10%'
         WHEN b.Desconto_Pct <= 0.15 THEN '3. 11% a 15%'
         ELSE '4. Acima de 15%' END AS Faixa_Desconto,
    CASE WHEN b.Atraso_Dias = 0 THEN '1. No prazo'
         WHEN b.Atraso_Dias <= 2 THEN '2. 1 a 2 dias'
         WHEN b.Atraso_Dias <= 5 THEN '3. 3 a 5 dias'
         ELSE '4. Mais de 5 dias' END AS Faixa_Atraso,
    CASE WHEN b.Status_Pedido = 'Cancelado' THEN '0. Cancelado'
         WHEN b.Margem_Pct < 0 THEN '1. Prejuízo'
         WHEN b.Margem_Pct < 0.15 THEN '2. Baixa (até 15%)'
         WHEN b.Margem_Pct < 0.30 THEN '3. Média (15% a 30%)'
         ELSE '4. Alta (acima de 30%)' END AS Classificacao_Margem,
    -- outliers e qualidade
    CASE WHEN b.Quantidade > l.Limite_Quantidade THEN 'Sim' ELSE 'Não' END AS Outlier_Quantidade,
    CASE WHEN b.Status_Pedido <> 'Cancelado' AND b.Receita_Liquida > l.Limite_Receita THEN 'Sim' ELSE 'Não' END AS Outlier_Receita,
    CASE WHEN b.Status_Pedido = 'Em processamento' AND b.Data_Pedido < DATE_SUB(l.Data_Referencia, 30) THEN 'Sim' ELSE 'Não' END AS Status_Desatualizado,
    -- Pareto
    CASE WHEN a.Perc_Acumulado <= 0.80 THEN 'A' WHEN a.Perc_Acumulado <= 0.95 THEN 'B' ELSE 'C' END AS Classe_ABC
FROM base b
CROSS JOIN limites l
LEFT JOIN abc a ON b.Produto = a.Produto

In [0]:
%sql
-- 3.2 A tabela tratada ficou correta? (contagens das novas variáveis)
SELECT
    COUNT(*) AS Registros,
    SUM(Pedido_Valido) AS Pedidos_Validos,
    SUM(Flag_Cancelado) AS Cancelados,
    SUM(Flag_Prejuizo) AS Pedidos_Prejuizo,
    SUM(CASE WHEN Outlier_Receita = 'Sim' THEN 1 ELSE 0 END) AS Outliers_Receita,
    SUM(CASE WHEN Status_Desatualizado = 'Sim' THEN 1 ELSE 0 END) AS Status_Desatualizados,
    SUM(Periodo_Comparavel) AS Pedidos_Jan_Set,
    COUNT(DISTINCT CASE WHEN Classe_ABC = 'A' THEN Produto END) AS Produtos_Classe_A
FROM base_projeto_tratada

In [0]:
%sql
-- 3.3 View mensal com métricas temporais
CREATE OR REPLACE VIEW base_projeto_mensal AS
WITH mensal AS (
    SELECT
        Ano_Mes, Ano, Mes,
        SUM(Receita_Liquida) AS Receita,
        SUM(Lucro) AS Lucro,
        SUM(Meta_Valida) AS Meta,
        SUM(Pedido_Valido) AS Pedidos
    FROM base_projeto_tratada
    GROUP BY Ano_Mes, Ano, Mes
)
SELECT
    Ano_Mes, Ano, Mes,
    ROUND(Receita, 2) AS Receita,
    ROUND(Lucro, 2) AS Lucro,
    Pedidos,
    ROUND(Lucro * 100 / Receita, 2) AS Margem_Perc,
    ROUND(Receita * 100 / Meta, 2) AS Atingimento_Meta_Perc,
    ROUND((Receita / LAG(Receita) OVER (ORDER BY Ano_Mes) - 1) * 100, 2) AS Var_Mes_Anterior_Perc,
    ROUND((Receita / LAG(Receita, 12) OVER (ORDER BY Ano_Mes) - 1) * 100, 2) AS Var_Ano_Anterior_Perc,
    ROUND(AVG(Receita) OVER (ORDER BY Ano_Mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS Receita_Media_Movel_3M,
    ROUND(SUM(Lucro) OVER (ORDER BY Ano_Mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) * 100
        / SUM(Receita) OVER (ORDER BY Ano_Mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS Margem_Movel_3M_Perc,
    ROUND(SUM(Receita) OVER (PARTITION BY Ano ORDER BY Ano_Mes), 2) AS Receita_Acumulada_Ano,
    RANK() OVER (PARTITION BY Ano ORDER BY Receita DESC) AS Ranking_Mes_No_Ano
FROM mensal

In [0]:
# 3.4 Como a receita evolui mês a mês? (receita do mês x média móvel de 3 meses)
df = spark.sql("""
    SELECT *, DATE_FORMAT(Ano_Mes, 'yyyy-MM') AS Mes_Ano
    FROM base_projeto_mensal
    ORDER BY Ano_Mes
""").toPandas()
display(df)

df["Receita_Mi"] = df["Receita"] / 1e6
df["Media_Movel_3M_Mi"] = df["Receita_Media_Movel_3M"] / 1e6
df.plot(x="Mes_Ano", y=["Receita_Mi", "Media_Movel_3M_Mi"], color=[CINZA, AZUL], rot=90,
        xticks=range(len(df)), figsize=(12, 4), title="Receita líquida mensal e média móvel de 3 meses (R$ milhões)")
plt.show()

## 4. KPIs

In [0]:
%sql
-- 4.1 Quanto vale cada KPI em cada ano? (só jan-set, para comparar períodos iguais)
SELECT
    Ano,
    ROUND(SUM(Receita_Liquida), 2) AS Receita_Liquida,
    ROUND(SUM(Lucro), 2) AS Lucro,
    ROUND(SUM(Lucro) * 100 / SUM(Receita_Liquida), 2) AS Margem_Perc,
    ROUND(SUM(Receita_Liquida) * 100 / SUM(Meta_Valida), 2) AS Atingimento_Meta_Perc,
    ROUND(SUM(Receita_Liquida) / SUM(Pedido_Valido), 2) AS Ticket_Medio,
    ROUND(AVG(CASE WHEN Pedido_Valido = 1 THEN Satisfacao_Cliente END), 2) AS Satisfacao_Media,
    ROUND(SUM(Flag_Devolvido) * 100 / SUM(Pedido_Valido), 2) AS Taxa_Devolucao_Perc,
    ROUND(SUM(Flag_Cancelado) * 100 / COUNT(*), 2) AS Taxa_Cancelamento_Perc,
    ROUND(AVG(CASE WHEN Pedido_Valido = 1 THEN Atraso_Dias END), 2) AS Atraso_Medio_Dias
FROM base_projeto_tratada
WHERE Periodo_Comparavel = 1
GROUP BY Ano
ORDER BY Ano

## 5. Nível Estratégico – Diretoria (P1 a P4)

In [0]:
# P1. Como estão receita, lucro e margem de 2026 em relação ao mesmo período de 2025?
df = spark.sql("""
    SELECT
        Ano,
        SUM(Receita_Liquida) / 1e6 AS Receita_Mi,
        SUM(Lucro) / 1e6 AS Lucro_Mi,
        SUM(Lucro) * 100 / SUM(Receita_Liquida) AS Margem_Perc
    FROM base_projeto_tratada
    WHERE Periodo_Comparavel = 1
    GROUP BY Ano
    ORDER BY Ano
""").toPandas()

# variação em relação ao ano anterior
df["Var_Receita_Perc"] = df["Receita_Mi"].pct_change() * 100
df["Var_Lucro_Perc"] = df["Lucro_Mi"].pct_change() * 100
df["Var_Margem_pp"] = df["Margem_Perc"].diff()
display(df.round(2))

df.plot.bar(x="Ano", y=["Receita_Mi", "Lucro_Mi"], color=[CINZA, AZUL], rot=0,
            title="Receita e lucro de jan a set, por ano (R$ milhões)")
plt.show()

In [0]:
# P2. A rentabilidade está melhorando ou piorando ao longo dos meses?
df = spark.sql("""
    SELECT DATE_FORMAT(Ano_Mes, 'yyyy-MM') AS Mes_Ano, Margem_Perc, Margem_Movel_3M_Perc
    FROM base_projeto_mensal
    WHERE Ano >= 2025
    ORDER BY Mes_Ano
""").toPandas()
display(df)

df.plot(x="Mes_Ano", y=["Margem_Perc", "Margem_Movel_3M_Perc"], color=[CINZA, VERMELHO],
        marker="o", rot=90, xticks=range(len(df)), title="Margem mensal e média móvel de 3 meses (%)")
plt.show()

In [0]:
# P3. Estamos atingindo a meta de receita?
df = spark.sql("""
    SELECT Ano, SUM(Receita_Liquida) * 100 / SUM(Meta_Valida) AS Atingimento_Meta_Perc
    FROM base_projeto_tratada
    WHERE Periodo_Comparavel = 1
    GROUP BY Ano
    ORDER BY Ano
""").toPandas()
display(df.round(2))

ax = df.plot.bar(x="Ano", y="Atingimento_Meta_Perc", color=AZUL, rot=0, legend=False, ylim=(0, 110),
                 title="Atingimento da meta de receita, jan a set (%)")
ax.axhline(100, color=VERMELHO, linestyle="--")   # linha da meta
plt.show()

In [0]:
# P4. Qual região exige atenção imediata da Diretoria? (jan-set/2026 x jan-set/2025)
df = spark.sql("""
    SELECT
        Regiao,
        SUM(CASE WHEN Ano = 2025 THEN Receita_Liquida END) / 1e6 AS Receita_2025_Mi,
        SUM(CASE WHEN Ano = 2026 THEN Receita_Liquida END) / 1e6 AS Receita_2026_Mi,
        SUM(CASE WHEN Ano = 2025 THEN Lucro END) * 100 / SUM(CASE WHEN Ano = 2025 THEN Receita_Liquida END) AS Margem_2025,
        SUM(CASE WHEN Ano = 2026 THEN Lucro END) * 100 / SUM(CASE WHEN Ano = 2026 THEN Receita_Liquida END) AS Margem_2026
    FROM base_projeto_tratada
    WHERE Periodo_Comparavel = 1 AND Ano IN (2025, 2026)
    GROUP BY Regiao
""").toPandas()

df["Var_Receita_Perc"] = (df["Receita_2026_Mi"] / df["Receita_2025_Mi"] - 1) * 100
df["Var_Margem_pp"] = df["Margem_2026"] - df["Margem_2025"]
df = df.sort_values("Var_Receita_Perc")
display(df.round(2))

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.barh(df["Regiao"], df["Var_Receita_Perc"], color=[VERMELHO if v < 0 else AZUL for v in df["Var_Receita_Perc"]])
ax1.set_title("Variação da receita (%)")
ax2.barh(df["Regiao"], df["Var_Margem_pp"], color=VERMELHO)
ax2.set_title("Variação da margem (p.p.)")
plt.tight_layout()
plt.show()

## 6. Nível Gerencial – Gestores (P5 a P10)

In [0]:
# P5. Quais subcategorias explicam a queda da margem em 2026? (jan-set)
df = spark.sql("""
    SELECT
        Categoria,
        Subcategoria,
        SUM(CASE WHEN Ano = 2025 THEN Lucro END) * 100 / SUM(CASE WHEN Ano = 2025 THEN Receita_Liquida END) AS Margem_2025,
        SUM(CASE WHEN Ano = 2026 THEN Lucro END) * 100 / SUM(CASE WHEN Ano = 2026 THEN Receita_Liquida END) AS Margem_2026
    FROM base_projeto_tratada
    WHERE Periodo_Comparavel = 1
    GROUP BY Categoria, Subcategoria
""").toPandas()

df["Var_pp"] = df["Margem_2026"] - df["Margem_2025"]
df = df.sort_values("Var_pp", ascending=False)   # maiores quedas ficam no topo do gráfico
display(df.round(1))

df.plot.barh(x="Subcategoria", y=["Margem_2025", "Margem_2026"], color=[CINZA, VERMELHO],
             figsize=(10, 5), title="Margem por subcategoria, jan a set: 2025 x 2026 (%)")
plt.show()

In [0]:
# P6. Em quais estados a receita caiu? (jan-set/2026 x jan-set/2025)
df = spark.sql("""
    SELECT
        Regiao,
        Estado,
        SUM(CASE WHEN Ano = 2025 THEN Receita_Liquida END) AS Receita_2025,
        SUM(CASE WHEN Ano = 2026 THEN Receita_Liquida END) AS Receita_2026,
        SUM(CASE WHEN Ano = 2025 THEN Pedido_Valido END) AS Pedidos_2025,
        SUM(CASE WHEN Ano = 2026 THEN Pedido_Valido END) AS Pedidos_2026
    FROM base_projeto_tratada
    WHERE Periodo_Comparavel = 1 AND Ano IN (2025, 2026)
    GROUP BY Regiao, Estado
""").toPandas()

df["Var_Receita_Perc"] = (df["Receita_2026"] / df["Receita_2025"] - 1) * 100
df = df.sort_values("Var_Receita_Perc", ascending=False)
display(df.round(1))

plt.figure(figsize=(10, 5))
plt.barh(df["Estado"], df["Var_Receita_Perc"], color=[VERMELHO if v < 0 else AZUL for v in df["Var_Receita_Perc"]])
plt.title("Variação da receita por estado, jan a set: 2026 x 2025 (%)")
plt.show()

In [0]:
# P7. Quais segmentos de clientes ficam mais abaixo da meta e qual o papel do desconto?
df = spark.sql("""
    SELECT
        Segmento_Cliente,
        SUM(Pedido_Valido) AS Pedidos,
        AVG(CASE WHEN Pedido_Valido = 1 THEN Desconto_Pct END) * 100 AS Desconto_Medio_Perc,
        SUM(Lucro) * 100 / SUM(Receita_Liquida) AS Margem_Perc,
        SUM(Receita_Liquida) * 100 / SUM(Meta_Valida) AS Atingimento_Meta_Perc
    FROM base_projeto_tratada
    GROUP BY Segmento_Cliente
    ORDER BY Atingimento_Meta_Perc
""").toPandas()
display(df.round(2))

df.plot.bar(x="Segmento_Cliente", y=["Atingimento_Meta_Perc", "Desconto_Medio_Perc"], subplots=True,
            layout=(1, 2), legend=False, rot=0, color=[AZUL, VERMELHO],
            title=["Atingimento da meta (%)", "Desconto médio (%)"])
plt.show()

In [0]:
# P8. Quais canais têm maior taxa de devolução e de cancelamento?
df = spark.sql("""
    SELECT
        Canal_Venda,
        COUNT(*) AS Pedidos,
        SUM(Flag_Devolvido) * 100 / SUM(Pedido_Valido) AS Taxa_Devolucao_Perc,
        SUM(Flag_Cancelado) * 100 / COUNT(*) AS Taxa_Cancelamento_Perc,
        SUM(Receita_Cancelada) AS Receita_Cancelada
    FROM base_projeto_tratada
    GROUP BY Canal_Venda
    ORDER BY Taxa_Devolucao_Perc DESC
""").toPandas()
display(df.round(2))

df.plot.bar(x="Canal_Venda", y=["Taxa_Devolucao_Perc", "Taxa_Cancelamento_Perc"], color=[VERMELHO, CINZA],
            rot=0, title="Taxa de devolução e de cancelamento por canal (%)")
plt.show()

In [0]:
# P9. Quais regiões têm mais atraso e pior satisfação?
df = spark.sql("""
    SELECT
        Regiao,
        AVG(CASE WHEN Pedido_Valido = 1 THEN Atraso_Dias END) AS Atraso_Medio_Dias,
        AVG(CASE WHEN Pedido_Valido = 1 THEN Satisfacao_Cliente END) AS Satisfacao_Media,
        SUM(Flag_Devolvido) * 100 / SUM(Pedido_Valido) AS Taxa_Devolucao_Perc
    FROM base_projeto_tratada
    GROUP BY Regiao
    ORDER BY Atraso_Medio_Dias DESC
""").toPandas()
display(df.round(2))

df.plot.bar(x="Regiao", subplots=True, layout=(1, 3), legend=False, rot=45, color=[VERMELHO, AZUL, CINZA],
            figsize=(12, 4), title=["Atraso médio (dias)", "Satisfação média (1 a 5)", "Devolução (%)"])
plt.show()

In [0]:
# P10. O ranking de vendedores por receita é igual ao ranking por lucro?
df = spark.sql("""
    SELECT
        Vendedor,
        Equipe,
        SUM(Receita_Liquida) / 1e6 AS Receita_Mi,
        SUM(Lucro) / 1e6 AS Lucro_Mi,
        SUM(Lucro) * 100 / SUM(Receita_Liquida) AS Margem_Perc
    FROM base_projeto_tratada
    GROUP BY Vendedor, Equipe
""").toPandas()

df["Ranking_Receita"] = df["Receita_Mi"].rank(ascending=False).astype(int)
df["Ranking_Lucro"] = df["Lucro_Mi"].rank(ascending=False).astype(int)
df = df.sort_values("Ranking_Receita")
display(df.round(2))

# dispersão: quem está mais à direita vende mais; quem está mais acima lucra mais
ax = df.plot.scatter(x="Receita_Mi", y="Lucro_Mi", s=60, color=AZUL,
                     title="Receita x lucro por vendedor (R$ milhões)")
for _, v in df.iterrows():
    ax.annotate(v["Vendedor"], (v["Receita_Mi"], v["Lucro_Mi"]), fontsize=8)
plt.show()

## 7. Nível Analítico – Analistas (P11 a P18)

In [0]:
# P11. Quando o custo mudou e em quais categorias? (custo como % da receita bruta, mês a mês)
df = spark.sql("""
    SELECT
        DATE_FORMAT(Ano_Mes, 'yyyy-MM') AS Mes_Ano,
        Categoria,
        SUM(Custo_Total) * 100 / SUM(Receita_Bruta_Valida) AS Custo_Sobre_Receita_Perc
    FROM base_projeto_tratada
    WHERE Ano >= 2025 AND Pedido_Valido = 1
    GROUP BY Ano_Mes, Categoria
    ORDER BY Mes_Ano
""").toPandas()

tabela = df.pivot(index="Mes_Ano", columns="Categoria", values="Custo_Sobre_Receita_Perc").round(1)
display(tabela.reset_index())

tabela.plot(marker="o", rot=90, xticks=range(len(tabela)), title="Custo sobre a receita bruta, por mês e categoria (%)")
plt.show()

In [0]:
# P12. Em quais produtos o custo subiu mais do que o preço? (jan-set/2026 x jan-set/2025)
df = spark.sql("""
    SELECT
        Produto,
        Classe_ABC,
        AVG(CASE WHEN Ano = 2025 THEN Preco_Unitario END) AS Preco_2025,
        AVG(CASE WHEN Ano = 2026 THEN Preco_Unitario END) AS Preco_2026,
        SUM(CASE WHEN Ano = 2025 THEN Custo_Total END) / SUM(CASE WHEN Ano = 2025 THEN Quantidade END) AS Custo_Unit_2025,
        SUM(CASE WHEN Ano = 2026 THEN Custo_Total END) / SUM(CASE WHEN Ano = 2026 THEN Quantidade END) AS Custo_Unit_2026
    FROM base_projeto_tratada
    WHERE Pedido_Valido = 1 AND Periodo_Comparavel = 1 AND Ano IN (2025, 2026)
    GROUP BY Produto, Classe_ABC
""").toPandas()

df["Var_Preco_Perc"] = (df["Preco_2026"] / df["Preco_2025"] - 1) * 100
df["Var_Custo_Perc"] = (df["Custo_Unit_2026"] / df["Custo_Unit_2025"] - 1) * 100
df = df.sort_values("Var_Custo_Perc")
display(df.round(1))

df.plot.barh(x="Produto", y=["Var_Preco_Perc", "Var_Custo_Perc"], color=[CINZA, VERMELHO], figsize=(10, 5),
             title="Variação do preço x variação do custo unitário, 2026 x 2025 (%)")
plt.show()

In [0]:
# P13. A partir de qual faixa de desconto a margem despenca e surgem prejuízos?
df = spark.sql("""
    SELECT
        Faixa_Desconto,
        COUNT(*) AS Pedidos,
        SUM(Lucro) * 100 / SUM(Receita_Liquida) AS Margem_Perc,
        SUM(Flag_Prejuizo) AS Pedidos_Prejuizo
    FROM base_projeto_tratada
    WHERE Pedido_Valido = 1
    GROUP BY Faixa_Desconto
    ORDER BY Faixa_Desconto
""").toPandas()
display(df.round(2))

correlacao = spark.sql("SELECT CORR(Desconto_Pct, Margem_Pct) FROM base_projeto_tratada WHERE Pedido_Valido = 1").first()[0]
print(f"Correlação desconto x margem (pedido a pedido): {correlacao:.2f}")

cores = [CINZA, CINZA, CINZA, VERMELHO]
df.plot.bar(x="Faixa_Desconto", y="Margem_Perc", color=cores, rot=0, legend=False,
            title="Margem por faixa de desconto (%)")
plt.show()

In [0]:
# P14. Quais pedidos deram prejuízo e o que eles têm em comum? (chegando aos registros)
registros = spark.sql("""
    SELECT ID_Pedido, Data_Pedido, Ano, Estado, Canal_Venda, Segmento_Cliente, Produto, Vendedor,
           Quantidade, Desconto_Perc, Receita_Liquida, Custo_Total, Lucro, Margem_Perc
    FROM base_projeto_tratada
    WHERE Flag_Prejuizo = 1
    ORDER BY Lucro
""").toPandas()
display(registros)

print("Pedidos com prejuízo:", len(registros), "| Menor desconto entre eles:", registros["Desconto_Perc"].min(), "%")

pd.crosstab(registros["Produto"], registros["Ano"]).plot.barh(
    stacked=True, title="Pedidos com prejuízo por produto e ano")
plt.show()

In [0]:
# P15. A receita está concentrada em poucos produtos (Pareto)? Esses produtos têm margem menor?
df = spark.sql("""
    SELECT
        Produto,
        Classe_ABC,
        SUM(Receita_Liquida) / 1e6 AS Receita_Mi,
        SUM(Lucro) / 1e6 AS Lucro_Mi,
        SUM(Lucro) * 100 / SUM(Receita_Liquida) AS Margem_Perc
    FROM base_projeto_tratada
    GROUP BY Produto, Classe_ABC
    ORDER BY Receita_Mi DESC
""").toPandas()

df["Receita_Acumulada_Perc"] = df["Receita_Mi"].cumsum() / df["Receita_Mi"].sum() * 100
df["Participacao_Lucro_Perc"] = df["Lucro_Mi"] / df["Lucro_Mi"].sum() * 100
display(df.round(2))

# barras = receita de cada produto (cor pela classe); linha = % acumulado da receita
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df["Produto"], df["Receita_Mi"], color=df["Classe_ABC"].map({"A": AZUL, "B": CINZA, "C": "#d9d9d9"}))
ax.set_ylabel("Receita (R$ milhões)")
ax.tick_params(axis="x", rotation=60)
ax2 = ax.twinx()
ax2.plot(df["Produto"], df["Receita_Acumulada_Perc"], color=VERMELHO, marker="o")
ax2.axhline(80, color=VERMELHO, linestyle="--", linewidth=0.8)
ax2.set_ylabel("% acumulado da receita")
ax.set_title("Curva ABC dos produtos (azul = classe A)")
plt.show()

In [0]:
# P16. O atraso na entrega reduz a satisfação e aumenta a devolução?
df = spark.sql("""
    SELECT
        Faixa_Atraso,
        COUNT(*) AS Pedidos,
        AVG(Satisfacao_Cliente) AS Satisfacao_Media,
        SUM(Flag_Devolvido) * 100 / COUNT(*) AS Taxa_Devolucao_Perc
    FROM base_projeto_tratada
    WHERE Pedido_Valido = 1
    GROUP BY Faixa_Atraso
    ORDER BY Faixa_Atraso
""").toPandas()
display(df.round(2))

correlacao = spark.sql("SELECT CORR(Atraso_Dias, Satisfacao_Cliente) FROM base_projeto_tratada WHERE Pedido_Valido = 1").first()[0]
print(f"Correlação atraso x satisfação (pedido a pedido): {correlacao:.2f}")

df.plot.bar(x="Faixa_Atraso", y=["Satisfacao_Media", "Taxa_Devolucao_Perc"], subplots=True, layout=(1, 2),
            legend=False, rot=0, color=[AZUL, VERMELHO], figsize=(12, 4),
            title=["Satisfação média (1 a 5)", "Taxa de devolução (%)"])
plt.show()

In [0]:
# P17. Existe sazonalidade? A meta considera essa sazonalidade?
df = spark.sql("""
    SELECT
        Mes,
        AVG(Fator_Sazonal) AS Fator_Sazonal_Medio,
        SUM(Receita_Liquida) * 100 / SUM(Meta_Valida) AS Atingimento_Meta_Perc
    FROM base_projeto_tratada
    GROUP BY Mes
    ORDER BY Mes
""").toPandas()
display(df.round(2))

df.plot.bar(x="Mes", y=["Fator_Sazonal_Medio", "Atingimento_Meta_Perc"], subplots=True, layout=(1, 2),
            legend=False, rot=0, color=[CINZA, AZUL], figsize=(12, 4),
            title=["Fator de preço sazonal (1 = normal)", "Atingimento da meta (%)"])
plt.show()

In [0]:
# P18. Quais pedidos são valores extremos (outliers) e de quais segmentos vêm?
df = spark.sql("""
    SELECT
        Segmento_Cliente,
        COUNT(*) AS Pedidos,
        SUM(CASE WHEN Outlier_Quantidade = 'Sim' THEN 1 ELSE 0 END) AS Outliers_Quantidade,
        SUM(CASE WHEN Outlier_Receita = 'Sim' THEN 1 ELSE 0 END) AS Outliers_Receita,
        SUM(CASE WHEN Outlier_Receita = 'Sim' THEN Receita_Liquida ELSE 0 END) * 100 / SUM(Receita_Liquida) AS Receita_Dos_Outliers_Perc
    FROM base_projeto_tratada
    WHERE Pedido_Valido = 1
    GROUP BY Segmento_Cliente
    ORDER BY Outliers_Quantidade DESC
""").toPandas()
display(df.round(1))

df.plot.bar(x="Segmento_Cliente", y=["Outliers_Quantidade", "Outliers_Receita"], color=[AZUL, CINZA], rot=0,
            title="Pedidos outliers (regra IQR) por segmento")
plt.show()

## 8. Dashboard (3 páginas = 3 níveis)
As 6 células abaixo montam e publicam o dashboard. Se rodar de novo, ele é atualizado (não duplica).

In [0]:
# 8.1 Datasets do dashboard
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

TABELA = f"{CATALOGO}.{SCHEMA}.base_projeto_tratada"
NOME_DASHBOARD = "Projeto - Dashboard Multinivel de Vendas"
ANO_ATUAL, ANO_ANTERIOR = 2026, 2025

# Uma coluna por categoria: o gráfico de linhas só funcionou com uma medida por série (sem "cor por categoria")
CATEGORIAS = {"Tec": "Tecnologia", "Corp": "Corporativo", "Esc": "Escritório", "Casa": "Casa & Utilidades"}
colunas_categoria = ", ".join(
    f"CASE WHEN Categoria = '{nome}' THEN {coluna} ELSE 0 END AS {coluna}_{sigla}"
    for sigla, nome in CATEGORIAS.items()
    for coluna in ["Lucro", "Receita_Liquida", "Custo_Total", "Receita_Bruta_Valida"])

datasets = [
    {"name": "base", "displayName": "Base tratada (pedidos)",
     "queryLines": [f"SELECT *, {colunas_categoria} FROM {TABELA}"]},
    {"name": "kpi_ano", "displayName": "KPIs ano atual x anterior (jan-set)",
     "queryLines": [
        "SELECT Regiao, ",
        f"  SUM(CASE WHEN Ano = {ANO_ATUAL} THEN Receita_Liquida ELSE 0 END) AS Receita_Atual, ",
        f"  SUM(CASE WHEN Ano = {ANO_ANTERIOR} THEN Receita_Liquida ELSE 0 END) AS Receita_Anterior, ",
        f"  SUM(CASE WHEN Ano = {ANO_ATUAL} THEN Lucro ELSE 0 END) AS Lucro_Atual, ",
        f"  SUM(CASE WHEN Ano = {ANO_ANTERIOR} THEN Lucro ELSE 0 END) AS Lucro_Anterior, ",
        f"  SUM(CASE WHEN Ano = {ANO_ATUAL} THEN Meta_Valida ELSE 0 END) AS Meta_Atual, ",
        f"  SUM(CASE WHEN Ano = {ANO_ANTERIOR} THEN Meta_Valida ELSE 0 END) AS Meta_Anterior, ",
        f"  SUM(CASE WHEN Ano = {ANO_ATUAL} THEN Pedido_Valido ELSE 0 END) AS Pedidos_Atual, ",
        f"  SUM(CASE WHEN Ano = {ANO_ANTERIOR} THEN Pedido_Valido ELSE 0 END) AS Pedidos_Anterior, ",
        f"  SUM(CASE WHEN Ano = {ANO_ATUAL} AND Pedido_Valido = 1 THEN Satisfacao_Cliente ELSE 0 END) AS Satisfacao_Soma_Atual, ",
        f"  SUM(CASE WHEN Ano = {ANO_ANTERIOR} AND Pedido_Valido = 1 THEN Satisfacao_Cliente ELSE 0 END) AS Satisfacao_Soma_Anterior, ",
        f"  (SUM(CASE WHEN Ano = {ANO_ATUAL} THEN Receita_Liquida ELSE 0 END) "
        f"   / NULLIF(SUM(CASE WHEN Ano = {ANO_ANTERIOR} THEN Receita_Liquida ELSE 0 END), 0) - 1) * 100 AS Variacao_Receita_Perc ",
        f"FROM {TABELA} ",
        "WHERE Periodo_Comparavel = 1 ",
        "GROUP BY Regiao"]},
    {"name": "mensal", "displayName": "Mensal por região (janela 3 meses)",
     "queryLines": [
        "WITH m AS (",
        f"  SELECT Ano_Mes, Regiao, SUM(Receita_Liquida) AS Receita, SUM(Lucro) AS Lucro FROM {TABELA} GROUP BY Ano_Mes, Regiao",
        ") ",
        "SELECT Ano_Mes, Regiao, Receita, Lucro, ",
        "  SUM(Receita) OVER (PARTITION BY Regiao ORDER BY Ano_Mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS Receita_3M, ",
        "  SUM(Lucro) OVER (PARTITION BY Regiao ORDER BY Ano_Mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS Lucro_3M ",
        "FROM m"]},
    {"name": "anual", "displayName": "Jan-set por ano, região e categoria",
     "queryLines": [
        "SELECT Ano_Texto, Regiao, Categoria, SUM(Receita_Liquida) AS Receita, SUM(Lucro) AS Lucro ",
        f"FROM {TABELA} ",
        "WHERE Periodo_Comparavel = 1 ",
        "GROUP BY Ano_Texto, Regiao, Categoria"]},
]
print(f"{len(datasets)} datasets definidos a partir de {TABELA}")

In [0]:
# 8.2 Funções que montam cada tipo de widget (KPI, filtro, barra, linha, dispersão, texto)
def campo(nome, expressao):
    return {"name": nome, "expression": expressao}

def col(nome):
    return campo(nome, f"`{nome}`")

def dim(nome, rotulo=None):
    d = col(nome)
    d["rotulo"] = rotulo or nome.replace("_", " ")
    return d

def consulta(ds, campos, nome="main_query", desagregado=False):
    return [{"name": nome, "query": {"datasetName": ds, "fields": campos, "disaggregated": desagregado}}]

def moldura(titulo, descricao=""):
    m = {"showTitle": True, "title": titulo}
    if descricao:
        m.update({"showDescription": True, "description": descricao})
    return m

def posicao(x, y, largura, altura):
    return {"x": x, "y": y, "width": largura, "height": altura}

def texto(nome, conteudo, p):
    return {"widget": {"name": nome, "multilineTextboxSpec": {"lines": [conteudo]}}, "position": p}

def kpi(nome, titulo, ds, valor, p, formato=None, alvo=None):
    enc = {"value": {"fieldName": valor["name"], "displayName": titulo}}
    campos = [valor]
    if alvo:  # alvo = valor do ano anterior (o KPI mostra a comparação)
        enc["target"] = {"fieldName": alvo["name"], "displayName": "Ano anterior"}
        campos.append(alvo)
    if formato:
        for e in enc.values():
            e["format"] = formato
    return {"widget": {"name": nome, "queries": consulta(ds, campos),
                       "spec": {"version": 2, "widgetType": "counter", "encodings": enc,
                                "frame": moldura(titulo)}},
            "position": p}

def filtro(nome, tipo, coluna, titulo, datasets_alvo, p):
    queries, fields = [], []
    for ds in datasets_alvo:
        q = f"{nome}_{ds}"
        queries += consulta(ds, [col(coluna)], q)
        fields.append({"fieldName": coluna, "displayName": titulo, "queryName": q})
    return {"widget": {"name": nome, "queries": queries,
                       "spec": {"version": 2, "widgetType": tipo, "encodings": {"fields": fields},
                                "frame": moldura(titulo)}},
            "position": p}

def barra(nome, titulo, descricao, ds, dimensao, medida, rotulo, p, horizontal=False,
          cor=None, agrupado=False, ordem="valor", formato=None):
    sort = {"by": ("x-reversed" if horizontal else "y-reversed") if ordem == "valor" else "natural-order"}
    cat = {"fieldName": dimensao["name"], "scale": {"type": "categorical", "sort": sort},
           "displayName": dimensao.get("rotulo", dimensao["name"])}
    val = {"fieldName": medida["name"], "scale": {"type": "quantitative"}, "displayName": rotulo}
    if formato:
        val["format"] = formato
    enc = {"x": val, "y": cat} if horizontal else {"x": cat, "y": val}
    campos = [campo(dimensao["name"], dimensao["expression"]), medida]
    if cor:
        enc["color"] = {"fieldName": cor["name"], "scale": {"type": "categorical"},
                        "displayName": cor.get("rotulo", cor["name"])}
        campos.append(campo(cor["name"], cor["expression"]))
    spec = {"version": 3, "widgetType": "bar", "encodings": enc, "frame": moldura(titulo, descricao)}
    if agrupado:
        spec["mark"] = {"layout": "group"}
    return {"widget": {"name": nome, "queries": consulta(ds, campos), "spec": spec}, "position": p}

def linha(nome, titulo, descricao, ds, x, medidas, p):
    # medidas = lista de (campo, rótulo); cada medida vira uma linha do gráfico
    y = {"scale": {"type": "quantitative"},
         "fields": [{"fieldName": m["name"], "displayName": r} for m, r in medidas]}
    enc = {"x": {"fieldName": x["name"], "scale": {"type": "temporal"}, "displayName": "Mês"}, "y": y}
    return {"widget": {"name": nome, "queries": consulta(ds, [x] + [m for m, _ in medidas]),
                       "spec": {"version": 3, "widgetType": "line", "encodings": enc,
                                "frame": moldura(titulo, descricao)}},
            "position": p}

def dispersao(nome, titulo, descricao, ds, x, y, cor, p):
    enc = {"x": {"fieldName": x[0], "scale": {"type": "quantitative"}, "displayName": x[1]},
           "y": {"fieldName": y[0], "scale": {"type": "quantitative"}, "displayName": y[1]},
           "color": {"fieldName": cor, "scale": {"type": "categorical"}, "displayName": cor}}
    return {"widget": {"name": nome, "queries": consulta(ds, [col(x[0]), col(y[0]), col(cor)], desagregado=True),
                       "spec": {"version": 3, "widgetType": "scatter", "encodings": enc,
                                "frame": moldura(titulo, descricao)}},
            "position": p}

# Formatos e medidas reutilizadas nas três páginas
REAIS = {"type": "number-currency", "currencyCode": "BRL", "abbreviation": "compact",
         "decimalPlaces": {"type": "max", "places": 2}}
NUM1 = {"type": "number", "decimalPlaces": {"type": "fixed", "places": 1}}
NUM2 = {"type": "number", "decimalPlaces": {"type": "fixed", "places": 2}}

RECEITA = campo("receita", "SUM(`Receita_Liquida`)")
LUCRO = campo("lucro", "SUM(`Lucro`)")
MARGEM = campo("margem", "SUM(`Lucro`) * 100 / NULLIF(SUM(`Receita_Liquida`), 0)")
META = campo("meta_perc", "SUM(`Receita_Liquida`) * 100 / NULLIF(SUM(`Meta_Valida`), 0)")
TICKET = campo("ticket", "SUM(`Receita_Liquida`) / NULLIF(SUM(`Pedido_Valido`), 0)")
DEVOL = campo("devolucao", "SUM(`Flag_Devolvido`) * 100 / NULLIF(SUM(`Pedido_Valido`), 0)")
ATRASO = campo("atraso", "SUM(`Atraso_Dias` * `Pedido_Valido`) / NULLIF(SUM(`Pedido_Valido`), 0)")
SATISF = campo("satisfacao", "SUM(`Satisfacao_Cliente` * `Pedido_Valido`) / NULLIF(SUM(`Pedido_Valido`), 0)")
CUSTO_RB = campo("custo_rb", "SUM(`Custo_Total`) * 100 / NULLIF(SUM(`Receita_Bruta_Valida`), 0)")
PEDIDOS = campo("pedidos", "SUM(`Pedido_Valido`)")
MES = col("Ano_Mes")   # nome igual ao da coluna: "mes" colidia com a coluna Mes no GROUP BY

# uma medida por categoria (usa as colunas por categoria criadas no dataset "base")
MARGEM_CAT = [(campo(f"margem_{s}", f"SUM(`Lucro_{s}`) * 100 / NULLIF(SUM(`Receita_Liquida_{s}`), 0)"), nome)
              for s, nome in CATEGORIAS.items()]
CUSTO_CAT = [(campo(f"custo_{s}", f"SUM(`Custo_Total_{s}`) * 100 / NULLIF(SUM(`Receita_Bruta_Valida_{s}`), 0)"), nome)
             for s, nome in CATEGORIAS.items()]
print("Funções e medidas prontas.")

In [0]:
# 8.3 Página 1 - Estratégico: 6 KPIs x ano anterior, 1 filtro, 4 gráficos + pontos de atenção
p1 = [
    texto("e_titulo",
          "## Nível 1 – Visão Estratégica (Diretoria)\n"
          "Como estamos? Comparação jan–set/2026 x jan–set/2025. Para investigar, siga para as páginas **Gerencial** e **Analítico**.",
          posicao(0, 0, 9, 2)),
    filtro("e_f_regiao", "filter-multi-select", "Regiao", "Região", ["kpi_ano", "mensal", "anual"], posicao(9, 0, 3, 2)),
    # linha de KPIs (valor atual x ano anterior)
    kpi("e_k_receita", "Receita líquida (jan–set)", "kpi_ano", campo("v", "SUM(`Receita_Atual`)"), posicao(0, 2, 2, 3),
        REAIS, campo("a", "SUM(`Receita_Anterior`)")),
    kpi("e_k_lucro", "Lucro (jan–set)", "kpi_ano", campo("v", "SUM(`Lucro_Atual`)"), posicao(2, 2, 2, 3),
        REAIS, campo("a", "SUM(`Lucro_Anterior`)")),
    kpi("e_k_margem", "Margem (%)", "kpi_ano", campo("v", "SUM(`Lucro_Atual`) * 100 / NULLIF(SUM(`Receita_Atual`), 0)"), posicao(4, 2, 2, 3),
        NUM1, campo("a", "SUM(`Lucro_Anterior`) * 100 / NULLIF(SUM(`Receita_Anterior`), 0)")),
    kpi("e_k_meta", "Atingimento da meta (%)", "kpi_ano", campo("v", "SUM(`Receita_Atual`) * 100 / NULLIF(SUM(`Meta_Atual`), 0)"), posicao(6, 2, 2, 3),
        NUM1, campo("a", "SUM(`Receita_Anterior`) * 100 / NULLIF(SUM(`Meta_Anterior`), 0)")),
    kpi("e_k_ticket", "Ticket médio", "kpi_ano", campo("v", "SUM(`Receita_Atual`) / NULLIF(SUM(`Pedidos_Atual`), 0)"), posicao(8, 2, 2, 3),
        REAIS, campo("a", "SUM(`Receita_Anterior`) / NULLIF(SUM(`Pedidos_Anterior`), 0)")),
    kpi("e_k_satisf", "Satisfação média (1 a 5)", "kpi_ano", campo("v", "SUM(`Satisfacao_Soma_Atual`) / NULLIF(SUM(`Pedidos_Atual`), 0)"), posicao(10, 2, 2, 3),
        NUM2, campo("a", "SUM(`Satisfacao_Soma_Anterior`) / NULLIF(SUM(`Pedidos_Anterior`), 0)")),
    # tendência (P2) + pontos de atenção
    linha("e_l_margem", "Margem mensal e média móvel de 3 meses (%)",
          "A margem saiu da faixa de 22-24% e passou a rodar abaixo de 20% a partir de jan/2026.",
          "mensal", MES,
          [(campo("margem_mes", "SUM(`Lucro`) * 100 / NULLIF(SUM(`Receita`), 0)"), "Margem do mês"),
           (campo("margem_3m", "SUM(`Lucro_3M`) * 100 / NULLIF(SUM(`Receita_3M`), 0)"), "Média móvel 3 meses")],
          posicao(0, 5, 8, 6)),
    texto("e_atencao",
          "### Pontos de atenção\n"
          "- 🔴 **Margem caiu 4,7 p.p.** (22,8% → 18,2%) com receita estável (−1,6%): **lucro −21,7%**.\n"
          "- 🔴 A queda está em **Tecnologia e Corporativo**; Escritório e Casa seguem estáveis.\n"
          "- 🟠 **Sudeste** (41% da receita) recuou **11,4%**, puxado pelo ES (−32%).\n"
          "- 🟠 **Sul** cresceu 12%, mas a margem caiu 8,2 p.p.\n\n"
          "**Decisão:** priorizar a recuperação da margem de Tecnologia e Corporativo (custo e preço).",
          posicao(8, 5, 4, 6)),
    # comparações (P1, P5, P4)
    {"widget": {"name": "e_b_ano",
                "queries": consulta("anual", [col("Ano_Texto"), campo("receita", "SUM(`Receita`)"), campo("lucro", "SUM(`Lucro`)")]),
                "spec": {"version": 3, "widgetType": "bar",
                         "encodings": {"x": {"fieldName": "Ano_Texto", "scale": {"type": "categorical", "sort": {"by": "natural-order"}}, "displayName": "Ano"},
                                       "y": {"scale": {"type": "quantitative"}, "format": REAIS,
                                             "fields": [{"fieldName": "receita", "displayName": "Receita"},
                                                        {"fieldName": "lucro", "displayName": "Lucro"}]}},
                         "mark": {"layout": "group"},
                         "frame": moldura("Receita e lucro de jan a set, por ano",
                                          "Receita estável em 2026, mas o lucro recuou ao menor nível da série.")}},
     "position": posicao(0, 11, 4, 6)},
    barra("e_b_cat", "Margem por categoria e ano, jan a set (%)",
          "Tecnologia e Corporativo perdem cerca de 6 p.p. em 2026.", "anual",
          dim("Categoria"), campo("margem", "SUM(`Lucro`) * 100 / NULLIF(SUM(`Receita`), 0)"), "Margem (%)",
          posicao(4, 11, 4, 6), cor=dim("Ano_Texto", "Ano"), agrupado=True, ordem="natural"),
    barra("e_b_reg", "Variação da receita por região: jan–set/26 x jan–set/25 (%)",
          "Sudeste, maior região, é a que mais caiu; Centro-Oeste e Sul cresceram.", "kpi_ano",
          dim("Regiao", "Região"), campo("variacao", "SUM(`Variacao_Receita_Perc`)"), "Variação (%)",
          posicao(8, 11, 4, 6), formato=NUM1),
]
print(f"Página Estratégica: {len(p1)} widgets")

In [0]:
# 8.4 Página 2 - Gerencial: 5 filtros, 6 KPIs e 8 gráficos em 4 linhas temáticas
B = ["base"]
p2 = [
    texto("g_titulo",
          "## Nível 2 – Visão Gerencial (Gestores)\n"
          "Onde e por que está acontecendo? Use os filtros e clique nos gráficos para segmentar.",
          posicao(0, 0, 12, 2)),
    filtro("g_f_periodo", "filter-date-range-picker", "Data_Pedido", "Período", B, posicao(0, 2, 4, 1)),
    filtro("g_f_regiao", "filter-multi-select", "Regiao", "Região", B, posicao(4, 2, 2, 1)),
    filtro("g_f_categoria", "filter-multi-select", "Categoria", "Categoria", B, posicao(6, 2, 2, 1)),
    filtro("g_f_canal", "filter-multi-select", "Canal_Venda", "Canal", B, posicao(8, 2, 2, 1)),
    filtro("g_f_segmento", "filter-multi-select", "Segmento_Cliente", "Segmento", B, posicao(10, 2, 2, 1)),
    kpi("g_k_receita", "Receita líquida", "base", RECEITA, posicao(0, 3, 2, 3), REAIS),
    kpi("g_k_margem", "Margem (%)", "base", MARGEM, posicao(2, 3, 2, 3), NUM1),
    kpi("g_k_meta", "Atingimento da meta (%)", "base", META, posicao(4, 3, 2, 3), NUM1),
    kpi("g_k_ticket", "Ticket médio", "base", TICKET, posicao(6, 3, 2, 3), REAIS),
    kpi("g_k_devol", "Taxa de devolução (%)", "base", DEVOL, posicao(8, 3, 2, 3), NUM1),
    kpi("g_k_atraso", "Atraso médio (dias)", "base", ATRASO, posicao(10, 3, 2, 3), NUM2),
    # Linha 1 - MARGEM (P5): quando e onde a margem caiu
    linha("g_l_cat", "Margem mensal por categoria (%)",
          "Tecnologia e Corporativo descolam para baixo a partir de jan/2026.",
          "base", MES, MARGEM_CAT, posicao(0, 6, 6, 7)),
    barra("g_b_subcat", "Margem por subcategoria (%)",
          "Infraestrutura e Notebooks têm as menores margens.", "base",
          dim("Subcategoria"), MARGEM, "Margem (%)", posicao(6, 6, 6, 7), horizontal=True),
    # Linha 2 - GEOGRAFIA (P4, P6): onde a receita caiu
    barra("g_b_regano", "Receita líquida por região e ano",
          "Compare o Sudeste em 2026 com os anos anteriores (2026 vai até setembro).", "base",
          dim("Regiao", "Região"), RECEITA, "Receita", posicao(0, 13, 6, 8),
          cor=dim("Ano_Texto", "Ano"), agrupado=True, formato=REAIS),
    barra("g_b_estado", "Receita líquida por estado",
          "Clique em um estado para filtrar os demais gráficos.", "base",
          dim("Estado"), RECEITA, "Receita", posicao(6, 13, 6, 8), horizontal=True, formato=REAIS),
    # Linha 3 - COMERCIAL (P7, P10): segmentos e vendedores
    barra("g_b_seg", "Atingimento da meta por segmento de cliente (%)",
          "Enterprise tem o maior desconto médio e o menor atingimento.", "base",
          dim("Segmento_Cliente", "Segmento"), META, "Atingimento (%)", posicao(0, 21, 6, 7)),
    barra("g_b_vend", "Ranking de vendedores por lucro",
          "A líder em receita (Ana Lima) é só a 8ª em lucro: margem menor.", "base",
          dim("Vendedor"), LUCRO, "Lucro", posicao(6, 21, 6, 7), horizontal=True, formato=REAIS),
    # Linha 4 - CLIENTE E OPERAÇÃO (P8, P9)
    barra("g_b_canal", "Taxa de devolução por canal (%)",
          "Marketplace devolve mais; produto com defeito é o principal motivo.", "base",
          dim("Canal_Venda", "Canal"), DEVOL, "Devolução (%)", posicao(0, 28, 6, 6)),
    barra("g_b_atraso", "Atraso médio por região (dias)",
          "Nordeste e Norte atrasam o dobro do Sudeste e do Sul.", "base",
          dim("Regiao", "Região"), ATRASO, "Atraso (dias)", posicao(6, 28, 6, 6)),
]
print(f"Página Gerencial: {len(p2)} widgets")

In [0]:
# 8.5 Página 3 - Analítico: 12 filtros, 4 KPIs, 8 gráficos em 4 linhas temáticas
p3 = [
    texto("a_titulo",
          "## Nível 3 – Visão Analítica (Analistas)\n"
          "Quais dados explicam o fenômeno? Cruze dimensões, procure padrões e outliers; na dispersão cada ponto é um pedido.",
          posicao(0, 0, 12, 2)),
    filtro("a_f_periodo", "filter-date-range-picker", "Data_Pedido", "Período", B, posicao(0, 2, 2, 1)),
    filtro("a_f_regiao", "filter-multi-select", "Regiao", "Região", B, posicao(2, 2, 2, 1)),
    filtro("a_f_estado", "filter-multi-select", "Estado", "Estado", B, posicao(4, 2, 2, 1)),
    filtro("a_f_categoria", "filter-multi-select", "Categoria", "Categoria", B, posicao(6, 2, 2, 1)),
    filtro("a_f_subcat", "filter-multi-select", "Subcategoria", "Subcategoria", B, posicao(8, 2, 2, 1)),
    filtro("a_f_produto", "filter-multi-select", "Produto", "Produto", B, posicao(10, 2, 2, 1)),
    filtro("a_f_canal", "filter-multi-select", "Canal_Venda", "Canal", B, posicao(0, 3, 2, 1)),
    filtro("a_f_segmento", "filter-multi-select", "Segmento_Cliente", "Segmento", B, posicao(2, 3, 2, 1)),
    filtro("a_f_vendedor", "filter-multi-select", "Vendedor", "Vendedor", B, posicao(4, 3, 2, 1)),
    filtro("a_f_status", "filter-multi-select", "Status_Pedido", "Status", B, posicao(6, 3, 2, 1)),
    filtro("a_f_desconto", "filter-multi-select", "Faixa_Desconto", "Faixa de desconto", B, posicao(8, 3, 2, 1)),
    filtro("a_f_margem", "filter-multi-select", "Classificacao_Margem", "Classificação da margem", B, posicao(10, 3, 2, 1)),
    kpi("a_k_pedidos", "Pedidos válidos", "base", PEDIDOS, posicao(0, 4, 3, 3)),
    kpi("a_k_prej", "Pedidos com prejuízo", "base", campo("prej", "SUM(`Flag_Prejuizo`)"), posicao(3, 4, 3, 3)),
    kpi("a_k_prej_valor", "Prejuízo total", "base", campo("prej_valor", "SUM(`Valor_Prejuizo`)"), posicao(6, 4, 3, 3), REAIS),
    kpi("a_k_outlier", "Pedidos outliers de receita", "base",
        campo("outl", "SUM(CASE WHEN `Outlier_Receita` = 'Sim' THEN 1 ELSE 0 END)"), posicao(9, 4, 3, 3)),
    # Linha 1 - CUSTO E DESCONTO (P11, P13)
    linha("a_l_custo", "Custo sobre a receita bruta, por mês e categoria (%)",
          "Salto abrupto em jan/2026 em Tecnologia e Corporativo: custo subiu mais que o preço.",
          "base", MES, CUSTO_CAT, posicao(0, 7, 6, 7)),
    dispersao("a_d_desc", "Desconto x margem, pedido a pedido",
              "Correlação de -0,36. Os pontos abaixo de zero são pedidos com prejuízo (outliers).",
              "base", ("Desconto_Perc", "Desconto (%)"), ("Margem_Perc", "Margem (%)"), "Categoria", posicao(6, 7, 6, 7)),
    # Linha 2 - MARGEM E PREJUÍZO (P13, P14)
    barra("a_b_faixa_desc", "Margem por faixa de desconto (%)",
          "Acima de 15% de desconto a margem cai para menos da metade (26,8% para 12,0%).", "base",
          dim("Faixa_Desconto", "Faixa de desconto"), MARGEM, "Margem (%)", posicao(0, 14, 6, 6), ordem="natural"),
    barra("a_b_classif", "Pedidos por classificação de margem e ano",
          "Pedidos de margem baixa e com prejuízo crescem em 2026.", "base",
          dim("Classificacao_Margem", "Classificação"), campo("qtd_pedidos", "COUNT(`ID_Pedido`)"), "Pedidos",
          posicao(6, 14, 6, 6), cor=dim("Ano_Texto", "Ano"), agrupado=True, ordem="natural"),
    # Linha 3 - ENTREGA E EXPERIÊNCIA (P16)
    barra("a_b_satisf", "Satisfação média por faixa de atraso",
          "Cada dia de atraso reduz a satisfação (correlação de -0,57).", "base",
          dim("Faixa_Atraso", "Atraso"), SATISF, "Satisfação", posicao(0, 20, 6, 6), ordem="natural", formato=NUM2),
    barra("a_b_devol", "Taxa de devolução por faixa de atraso (%)",
          "Pedidos com mais de 5 dias de atraso devolvem 5 vezes mais.", "base",
          dim("Faixa_Atraso", "Atraso"), DEVOL, "Devolução (%)", posicao(6, 20, 6, 6), ordem="natural"),
    # Linha 4 - CONCENTRAÇÃO E DEVOLUÇÃO (P15, P8)
    barra("a_b_abc", "Curva ABC: receita líquida por produto",
          "6 produtos (classe A) = 77% da receita, todos com margem abaixo da média.", "base",
          dim("Produto"), RECEITA, "Receita", posicao(0, 26, 6, 8), horizontal=True,
          cor=dim("Classe_ABC", "Classe ABC"), formato=REAIS),
    barra("a_b_motivo", "Motivos de devolução por canal (pedidos)",
          "Filtre um canal para ver o motivo predominante.", "base",
          dim("Canal_Venda", "Canal"), campo("devolvidos", "SUM(`Flag_Devolvido`)"), "Pedidos devolvidos",
          posicao(6, 26, 6, 8), cor=dim("Motivo_Devolucao", "Motivo")),
]
print(f"Página Analítica: {len(p3)} widgets")

In [0]:
# 8.6 Junta as 3 páginas, cria (ou atualiza) o dashboard e publica
dashboard = {
    "datasets": datasets,
    "pages": [
        {"name": "estrategico", "displayName": "1. Estratégico - Diretoria", "pageType": "PAGE_TYPE_CANVAS", "layout": p1},
        {"name": "gerencial", "displayName": "2. Gerencial - Gestores", "pageType": "PAGE_TYPE_CANVAS", "layout": p2},
        {"name": "analitico", "displayName": "3. Analítico - Analistas", "pageType": "PAGE_TYPE_CANVAS", "layout": p3},
    ],
}

warehouse_id = next(iter(w.warehouses.list())).id
pasta = f"/Users/{w.current_user.me().user_name}"
corpo = {"display_name": NOME_DASHBOARD, "warehouse_id": warehouse_id,
         "serialized_dashboard": json.dumps(dashboard, ensure_ascii=False)}

existentes = w.api_client.do("GET", "/api/2.0/lakeview/dashboards", query={"page_size": 100}).get("dashboards", [])
existente = next((d for d in existentes if d.get("display_name") == NOME_DASHBOARD), None)

if existente:
    dashboard_id = existente["dashboard_id"]
    w.api_client.do("PATCH", f"/api/2.0/lakeview/dashboards/{dashboard_id}", body=corpo)
    print("Dashboard atualizado.")
else:
    dashboard_id = w.api_client.do("POST", "/api/2.0/lakeview/dashboards", body={**corpo, "parent_path": pasta})["dashboard_id"]
    print("Dashboard criado.")

w.api_client.do("POST", f"/api/2.0/lakeview/dashboards/{dashboard_id}/published",
                body={"warehouse_id": warehouse_id, "embed_credentials": True})

url = f"{w.config.host.rstrip('/')}/dashboardsv3/{dashboard_id}/published"
displayHTML(f"""
<div style="margin-top: 15px; padding: 12px 16px; border-radius: 8px; background-color: #f5f5f5; border: 1px solid #ddd;">
  <a href="{url}" target="_blank" style="font-weight: 600; text-decoration: none;">{NOME_DASHBOARD}</a>
</div>
""")